# RSNA Knee Abnormality Detection V03: Hierarchical State-Prior Weak-Label 2.5D Baseline

This notebook implements the complete pipeline from the official CSV and DICOM files to `submission.csv`:

1. Generate fold-safe calibrated soft targets from 58 gold-label studies and report-based rules.
2. Select one MRI series per study for the Sagittal, Coronal, and Axial planes.
3. Sort DICOM slices by spatial position, sample them uniformly, and construct 2.5D triplets.
4. Predict 12 labels with a shared EfficientNet-B0, within-plane pooling, and three-plane fusion.
5. Run study-level cross-validation, save checkpoints, infer the test set, and create a submission.

The configuration trains all five folds and ensembles their best checkpoints at inference.
Set `folds_to_train = (0,)` to verify the full pipeline on a single fold first.

> Important: only 58 of the 4,407 training studies have complete human labels.
V03 keeps the V01/V02 image pipeline and training configuration unchanged and only
redesigns weak-label calibration to fix the V02 signal collapse (out-of-fold
predictions compressed to a near-constant per-label base rate, mean std ~0.05):

- fit rule calibration only on the training gold rows of each fold (unchanged);
- shrink each report state toward a pooled, **state-specific** prior mean
  (positive > unmentioned > explicit_negative) instead of the single per-label
  prevalence, repairing the V02 target reversals and zero-support collapse;
- enforce state ordering with a minimum margin and conservative per-state ranges;
- drop the 50% confidence floor so zero-support states keep only 15% of their weight;
- keep gold weight 8 and no class `pos_weight`;
- report a best reference-AUC checkpoint and a prediction-spread collapse diagnostic.



In [ ]:
from __future__ import annotations

import gc
import json
import math
import os
import pickle
import random
import re
import time
import unicodedata
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import cv2
import numpy as np
import pandas as pd
import pydicom
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)
NOTEBOOK_STARTED_AT = time.monotonic()

LABELS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]
PLANES = ("Sagittal", "Coronal", "Axial")
UID = "StudyInstanceUID"


@dataclass
class CFG:
    version: str = "v03"
    seed: int = 2026
    image_size: int = 320
    samples_per_plane: int = 6
    triplet_gap: int = 2
    backbone: str = "efficientnet_b0"
    pretrained: bool = True
    local_backbone_weights: Optional[str] = None

    n_folds: int = 5
    folds_to_train: Tuple[int, ...] = (0, 1, 2, 3, 4)
    epochs: int = 4
    batch_size: int = 1
    accumulation_steps: int = 4
    num_workers: int = 2
    learning_rate: float = 2e-4
    weight_decay: float = 1e-4
    dropout: float = 0.30
    max_grad_norm: float = 5.0
    early_stopping_patience: int = 2

    runtime_limit_hours: float = 9.0
    runtime_safety_margin_minutes: float = 35.0
    auto_resume: bool = True
    resume_checkpoint: Optional[str] = None

    gold_weight: float = 8.0
    weak_positive_weight: float = 0.60
    weak_explicit_negative_weight: float = 0.45
    weak_unmentioned_weight: float = 0.03
    calibration_prior_strength: float = 6.0
    calibration_support_scale: float = 8.0
    calibration_prior_confidence: float = 0.15
    calibration_positive_margin: float = 0.10
    calibration_negative_margin: float = 0.10
    calibration_positive_range: Tuple[float, float] = (0.55, 0.95)
    calibration_unmentioned_range: Tuple[float, float] = (0.05, 0.80)
    calibration_negative_range: Tuple[float, float] = (0.01, 0.40)

    debug: bool = False
    debug_studies: int = 160


cfg = CFG()


def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


seed_everything(cfg.seed)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = DEVICE.type == "cuda"

print("Python/PyTorch:", torch.__version__)
print("pydicom:", pydicom.__version__)
print("scikit-learn:", sklearn.__version__)
print("Device:", DEVICE)
print(json.dumps(asdict(cfg), indent=2, ensure_ascii=False))



## 1. Locate the Kaggle data and load the CSV files

The confirmed competition directory structure is:

```text
rsna-knee-abnormality-detection/
├── train.csv
├── train_series.csv
├── train_series/{StudyInstanceUID}/{SeriesInstanceUID}/*.dcm
├── test.csv
├── test_series.csv
├── test_series/{StudyInstanceUID}/{SeriesInstanceUID}/*.dcm
└── sample_submission.csv
```



In [ ]:
def find_input_root() -> Path:
    candidates = [
        Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
        Path("/kaggle/input/rsna-knee-abnormality-detection"),
        Path.cwd(),
        Path.cwd() / "data",
    ]
    for root in candidates:
        if (root / "train.csv").exists() and (root / "train_series.csv").exists():
            return root

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for csv_path in kaggle_input.rglob("train_series.csv"):
            root = csv_path.parent
            if (root / "train.csv").exists():
                return root

    raise FileNotFoundError(
        "Competition data was not found. Add the RSNA Knee Abnormality Detection "
        "competition data as a Kaggle Notebook input."
    )


INPUT_ROOT = find_input_root()
TRAIN_IMAGE_ROOT = INPUT_ROOT / "train_series"
TEST_IMAGE_ROOT = INPUT_ROOT / "test_series"
WORK_DIR = Path("/kaggle/working/rsna_knee_v03")
if not Path("/kaggle/working").exists():
    WORK_DIR = Path.cwd() / "working" / "rsna_knee_v03"
WORK_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(INPUT_ROOT / "train.csv")
train_series_df = pd.read_csv(INPUT_ROOT / "train_series.csv")
test_df = pd.read_csv(INPUT_ROOT / "test.csv")
test_series_df = pd.read_csv(INPUT_ROOT / "test_series.csv")
sample_submission = pd.read_csv(INPUT_ROOT / "sample_submission.csv")

for label in LABELS:
    train_df[label] = pd.to_numeric(train_df[label], errors="coerce")

assert train_df[UID].is_unique, "StudyInstanceUID must be unique in train.csv"
assert test_df[UID].is_unique, "StudyInstanceUID must be unique in test.csv"
assert set(LABELS).issubset(sample_submission.columns)
assert TRAIN_IMAGE_ROOT.exists(), f"Training image directory does not exist: {TRAIN_IMAGE_ROOT}"
assert TEST_IMAGE_ROOT.exists(), f"Test image directory does not exist: {TEST_IMAGE_ROOT}"

print("INPUT_ROOT:", INPUT_ROOT)
print("WORK_DIR:", WORK_DIR)
print("Train studies / series:", len(train_df), len(train_series_df))
print("Test studies / series:", len(test_df), len(test_series_df))
display(train_series_df.head())



## 2. Generate reproducible weak labels from reports

Rows with complete human labels use official binary targets with weight 8. For every training
fold, multilingual report-rule states are calibrated only against that fold's training gold rows.
Hierarchical empirical-Bayes smoothing shrinks each state toward a pooled, state-specific prior
mean (estimated across all 12 labels from the fold's training gold rows), then enforces the
ordering positive > unmentioned > explicit_negative. Unmentioned findings receive very low
confidence instead of becoming hard negatives, and zero-support states keep only a small prior
floor. `assert_calibration_valid` blocks training if any label produces non-ordered or degenerate
targets.

The calibration improves V01/V02 supervision while preserving a leakage-safe gold validation
subset. These rules remain a reproducible baseline, not a clinical-grade NLP labeler.



In [ ]:
def normalize_report(text: object) -> str:
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.replace("\r", "\n")
    text = re.sub(r"[\t ]+", " ", text)
    return text


TARGET_PATTERNS: Dict[str, List[str]] = {
    "ACL": [
        r"\bacl\b", r"anterior cruciate ligament", r"anterior cruciate",
        r"ligamento cruzado anterior", r"ligament croise anterieur", r"\blca\b",
        r"voorste kruisband", r"\bvkb\b", r"vorder(?:e|es|en) kreuzband",
        r"legamento crociato anteriore",
    ],
    "MCL": [
        r"\bmcl\b", r"medial collateral ligament", r"medial collateral",
        r"ligamento colateral medial", r"ligamento colateral interno",
        r"ligament collateral medial", r"mediale collateralband", r"mediale collaterale band",
        r"innenband", r"legamento collaterale mediale",
    ],
    "Medial Meniscus": [
        r"medial menisc(?:us|al)", r"inner menisc(?:us|al)", r"internal menisc(?:us|al)",
        r"menisc(?:o|us) (?:medial|interno)", r"menisque (?:medial|interne)",
        r"mediale menisc", r"innenmenisk", r"menisco mediale",
    ],
    "Lateral Meniscus": [
        r"lateral menisc(?:us|al)", r"outer menisc(?:us|al)", r"external menisc(?:us|al)",
        r"menisc(?:o|us) (?:lateral|externo)", r"menisque (?:lateral|externe)",
        r"laterale menisc", r"aussenmenisk", r"menisco laterale",
    ],
    "Medial OA": [
        r"medial (?:tibiofemoral|femorotibial|femoro[- ]?tibial|joint|compartment)[^.;\n]{0,45}(?:osteoarthr|osteoarthros|arthros|artros|gonarthros|degenerat)",
        r"(?:osteoarthr|osteoarthros|arthros|artros|gonarthros|degenerat)[^.;\n]{0,45}medial (?:tibiofemoral|femorotibial|femoro[- ]?tibial|joint|compartment)",
        r"(?:artrosis|arthrose|artrose) femorotibial medial",
        r"medial compartment (?:oa|joint space narrowing)",
    ],
    "Lateral OA": [
        r"lateral (?:tibiofemoral|femorotibial|femoro[- ]?tibial|joint|compartment)[^.;\n]{0,45}(?:osteoarthr|osteoarthros|arthros|artros|gonarthros|degenerat)",
        r"(?:osteoarthr|osteoarthros|arthros|artros|gonarthros|degenerat)[^.;\n]{0,45}lateral (?:tibiofemoral|femorotibial|femoro[- ]?tibial|joint|compartment)",
        r"(?:artrosis|arthrose|artrose) femorotibial lateral",
        r"lateral compartment (?:oa|joint space narrowing)",
    ],
    "PF OA": [
        r"patello?femoral[^.;\n]{0,45}(?:osteoarthr|osteoarthros|arthros|artros|degenerat|chondropath|condropat)",
        r"(?:osteoarthr|osteoarthros|arthros|artros|degenerat|chondropath|condropat)[^.;\n]{0,45}patello?femoral",
        r"(?:retropatellar|patellar|rotulian|rotula)[^.;\n]{0,35}(?:chondropath|condropat|arthros|artros)",
        r"(?:chondropath|condropat|arthros|artros)[^.;\n]{0,35}(?:retropatellar|patellar|rotulian|rotula)",
    ],
    "Effusion": [
        r"joint effusion", r"knee effusion", r"\beffusion\b", r"\bhydrops\b",
        r"derrame (?:articular|de la articulacion)?", r"derrame articular",
        r"epanchement", r"gelenkerguss", r"versamento articolare", r"gewrichtsvocht",
    ],
    "Synovitis": [r"\bsynovitis\b", r"\bsinovitis\b", r"synoviale ontsteking", r"synovial inflammation"],
    "Baker's": [
        r"baker'?s? cyst", r"baker cyste", r"bakerzyste", r"bakerse cyste",
        r"popliteal cyst", r"quiste popliteo", r"kyste poplite", r"cisti poplitea",
    ],
    "Contusion": [
        r"bone (?:marrow )?(?:bruise|contusion)", r"osseous contusion", r"\bcontusion\b",
        r"contusion osseuse", r"contusion osea", r"contusione ossea", r"knochenkontusion",
        r"bone marrow edema", r"oedema osse", r"edema oseo",
    ],
    "Fracture": [
        r"\bfracture\b", r"\bfractura\b", r"\bfraktur\b", r"\bfrattura\b",
        r"\bbreuk\b", r"\bfissure osseuse\b",
    ],
}

NEGATION_BEFORE = re.compile(
    r"(?:\bno\b|\bnot\b|\bwithout\b|\babsence of\b|\babsent\b|\bnegative for\b|"
    r"\bsin\b|\bningun[ao]?\b|\bgeen\b|\bniet\b|\bohne\b|\bkein(?:e|en|er|es)?\b|"
    r"\bsans\b|\bpas de\b|\bsem\b|\bnao\b|\bsenza\b|\bnon\b)[^.;:\n]{0,65}$"
)
NEGATIVE_AFTER = re.compile(
    r"^[^.;:\n]{0,35}(?:\bintact\b|\bnormal\b|\bunremarkable\b|\bpreserved\b|"
    r"\bconserved\b|\bconservad[oa]\b|\bintegro\b|\bregelrecht\b|\bonopvallend\b|"
    r"\bwithout (?:tear|rupture|injury|abnormality)\b|\bno (?:tear|rupture|injury|abnormality)\b)"
)
COMPILED_PATTERNS = {
    label: [re.compile(pattern) for pattern in patterns]
    for label, patterns in TARGET_PATTERNS.items()
}


def classify_report_label(report: str, label: str) -> str:
    """Return positive, explicit_negative, or unmentioned."""
    text = normalize_report(report)
    if not text:
        return "unmentioned"

    has_negative = False
    for pattern in COMPILED_PATTERNS[label]:
        for match in pattern.finditer(text):
            before = text[max(0, match.start() - 80):match.start()]
            after = text[match.end():min(len(text), match.end() + 65)]
            negated = bool(NEGATION_BEFORE.search(before) or NEGATIVE_AFTER.search(after))
            if not negated:
                return "positive"
            has_negative = True
    return "explicit_negative" if has_negative else "unmentioned"


def build_report_statuses(frame: pd.DataFrame) -> pd.DataFrame:
    """Classify every report for every target, including the gold rows."""
    rows = []
    for _, row in tqdm(frame.iterrows(), total=len(frame), desc="Report rules"):
        record = {UID: row[UID]}
        report = row.get("Report", "")
        for label in LABELS:
            record[f"{label}__status"] = classify_report_label(report, label)
        rows.append(record)
    return pd.DataFrame(rows)


def build_reference_targets(frame: pd.DataFrame, statuses: pd.DataFrame) -> np.ndarray:
    """Build binary targets used only for fold stratification and rule diagnostics."""
    reference = np.zeros((len(frame), len(LABELS)), dtype=np.float32)
    gold_mask = frame[LABELS].notna().all(axis=1).to_numpy()
    for label_idx, label in enumerate(LABELS):
        reference[:, label_idx] = statuses[f"{label}__status"].eq("positive").to_numpy(np.float32)
        reference[gold_mask, label_idx] = frame.loc[gold_mask, label].to_numpy(np.float32)
    return reference


RULE_BASE_WEIGHTS = {
    "positive": cfg.weak_positive_weight,
    "explicit_negative": cfg.weak_explicit_negative_weight,
    "unmentioned": cfg.weak_unmentioned_weight,
}


def compute_global_state_means(
    frame: pd.DataFrame,
    statuses: pd.DataFrame,
    gold_indices: np.ndarray,
) -> Dict[str, float]:
    """Pool gold rows across all labels to estimate one robust prior mean per rule state.

    Per-label gold counts are tiny (often fewer than 25 studies), so individual
    (label, state) positive rates are noisy and occasionally reversed. In fold 0 the raw
    positive rate is below the unmentioned rate for Medial Meniscus and Effusion, and
    Lateral OA has zero positive/negative support. Pooling across the 12 labels gives a
    stable, correctly ordered semantic prior (positive > unmentioned > explicit_negative)
    that each label then shrinks toward, instead of shrinking every state to the single
    per-label prevalence as V02 did.
    """
    overall_prevalence = float(frame.iloc[gold_indices][LABELS].to_numpy(np.float32).mean())
    state_means: Dict[str, float] = {}
    for status in RULE_BASE_WEIGHTS:
        positives_total = 0.0
        support_total = 0
        for label in LABELS:
            gold_values = frame.iloc[gold_indices][label].to_numpy(np.float32)
            status_values = statuses.iloc[gold_indices][f"{label}__status"].to_numpy()
            mask = status_values == status
            support_total += int(mask.sum())
            positives_total += float(gold_values[mask].sum())
        state_means[status] = (
            positives_total / support_total if support_total else overall_prevalence
        )
    return state_means


def fit_rule_calibration(
    frame: pd.DataFrame,
    statuses: pd.DataFrame,
    calibration_indices: np.ndarray,
) -> Tuple[Dict[str, Dict[str, Tuple[float, float]]], pd.DataFrame]:
    """Fit hierarchical empirical-Bayes rule targets using training-fold gold rows only.

    V03 changes relative to V02:
    * Each (label, state) shrinks toward a pooled, state-specific prior mean instead of
      the single per-label prevalence, so explicit positives, explicit negatives, and
      unmentioned findings receive semantically distinct fallbacks. This repairs the V02
      target reversals and the degenerate zero-support labels whose three states
      collapsed to an identical value.
    * Range clamps plus an ordering margin guarantee positive > unmentioned >
      explicit_negative for every label.
    * Confidence weighting drops the old 50% floor; a zero-support state keeps only
      calibration_prior_confidence of its base weight instead of half.
    """
    complete_gold = frame[LABELS].notna().all(axis=1).to_numpy()
    calibration_indices = np.asarray(calibration_indices, dtype=np.int64)
    gold_indices = calibration_indices[complete_gold[calibration_indices]]
    if len(gold_indices) == 0:
        raise RuntimeError("No gold rows are available for fold-specific rule calibration.")

    state_means = compute_global_state_means(frame, statuses, gold_indices)
    pos_lo, pos_hi = cfg.calibration_positive_range
    unm_lo, unm_hi = cfg.calibration_unmentioned_range
    neg_lo, neg_hi = cfg.calibration_negative_range

    mapping: Dict[str, Dict[str, Tuple[float, float]]] = {}
    rows = []
    for label in LABELS:
        gold_values = frame.iloc[gold_indices][label].to_numpy(np.float32)
        prevalence = float(gold_values.mean())
        status_values = statuses.iloc[gold_indices][f"{label}__status"].to_numpy()

        raw_target: Dict[str, float] = {}
        support_by_status: Dict[str, int] = {}
        positives_by_status: Dict[str, float] = {}
        weight_by_status: Dict[str, float] = {}
        for status, base_weight in RULE_BASE_WEIGHTS.items():
            mask = status_values == status
            support = int(mask.sum())
            positives = float(gold_values[mask].sum()) if support else 0.0
            probability = (
                positives + cfg.calibration_prior_strength * state_means[status]
            ) / (support + cfg.calibration_prior_strength)
            support_reliability = support / (support + cfg.calibration_support_scale)
            reliability = (
                cfg.calibration_prior_confidence
                + (1.0 - cfg.calibration_prior_confidence) * support_reliability
            )
            raw_target[status] = float(probability)
            support_by_status[status] = support
            positives_by_status[status] = positives
            weight_by_status[status] = float(base_weight * reliability)

        # Clamp each state to a conservative range, then enforce the semantic ordering
        # positive >= unmentioned + positive_margin and explicit_negative <=
        # unmentioned - negative_margin so report evidence is never inverted by noise.
        unmentioned = float(np.clip(raw_target["unmentioned"], unm_lo, unm_hi))
        positive = float(np.clip(raw_target["positive"], pos_lo, pos_hi))
        negative = float(np.clip(raw_target["explicit_negative"], neg_lo, neg_hi))
        if positive < unmentioned + cfg.calibration_positive_margin:
            positive = min(pos_hi, unmentioned + cfg.calibration_positive_margin)
        if negative > unmentioned - cfg.calibration_negative_margin:
            negative = max(neg_lo, unmentioned - cfg.calibration_negative_margin)
        constrained = {
            "positive": positive,
            "unmentioned": unmentioned,
            "explicit_negative": negative,
        }

        mapping[label] = {}
        for status in RULE_BASE_WEIGHTS:
            mapping[label][status] = (constrained[status], weight_by_status[status])
            rows.append({
                "label": label,
                "status": status,
                "gold_support": support_by_status[status],
                "gold_positive": positives_by_status[status],
                "gold_prevalence": prevalence,
                "state_prior_mean": state_means[status],
                "raw_soft_target": raw_target[status],
                "soft_target": constrained[status],
                "confidence_weight": weight_by_status[status],
            })

    return mapping, pd.DataFrame(rows)


def assert_calibration_valid(calibration: pd.DataFrame) -> None:
    """Fail fast if any label yields non-ordered, degenerate, or over-confident targets."""
    prior_confidence = cfg.calibration_prior_confidence
    for label, group in calibration.groupby("label"):
        by_status = group.set_index("status")
        positive = float(by_status.loc["positive", "soft_target"])
        unmentioned = float(by_status.loc["unmentioned", "soft_target"])
        negative = float(by_status.loc["explicit_negative", "soft_target"])
        assert positive >= unmentioned >= negative, (
            f"{label}: state targets are not ordered "
            f"(positive={positive:.3f}, unmentioned={unmentioned:.3f}, negative={negative:.3f})"
        )
        assert not (abs(positive - unmentioned) < 1e-6 and abs(unmentioned - negative) < 1e-6), (
            f"{label}: all three state targets are identical"
        )
        for status in ("positive", "explicit_negative"):
            support = int(by_status.loc[status, "gold_support"])
            weight = float(by_status.loc[status, "confidence_weight"])
            floor = RULE_BASE_WEIGHTS[status] * prior_confidence
            if support == 0:
                assert weight <= floor + 1e-6, (
                    f"{label}/{status}: zero-support state keeps confidence {weight:.3f} "
                    f"above the prior floor {floor:.3f}"
                )


def build_fold_targets(
    frame: pd.DataFrame,
    statuses: pd.DataFrame,
    calibration_indices: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    """Create fold-safe soft targets and confidence weights for all studies."""
    mapping, calibration = fit_rule_calibration(frame, statuses, calibration_indices)
    targets_array = np.zeros((len(frame), len(LABELS)), dtype=np.float32)
    weights_array = np.zeros_like(targets_array)
    complete_gold = frame[LABELS].notna().all(axis=1).to_numpy()

    for label_idx, label in enumerate(LABELS):
        for row_idx, status in enumerate(statuses[f"{label}__status"]):
            if complete_gold[row_idx]:
                targets_array[row_idx, label_idx] = float(frame.iloc[row_idx][label])
                weights_array[row_idx, label_idx] = cfg.gold_weight
            else:
                probability, confidence_weight = mapping[label][status]
                targets_array[row_idx, label_idx] = probability
                weights_array[row_idx, label_idx] = confidence_weight

    return targets_array, weights_array, calibration


report_statuses = build_report_statuses(train_df)
is_gold = train_df[LABELS].notna().all(axis=1).to_numpy()
reference_targets = build_reference_targets(train_df, report_statuses)
targets = reference_targets.copy()

# This all-gold calibration is exported for inspection only. Training below always
# refits calibration from the training gold rows of the current fold.
diagnostic_targets, diagnostic_weights, diagnostic_calibration = build_fold_targets(
    train_df, report_statuses, np.arange(len(train_df))
)
diagnostic_calibration.to_csv(WORK_DIR / "v03_rule_calibration_diagnostic.csv", index=False)
assert_calibration_valid(diagnostic_calibration)

weak_label_export = train_df[[UID, "Report"]].copy()
weak_label_export["is_gold"] = is_gold
for idx, label in enumerate(LABELS):
    weak_label_export[label] = diagnostic_targets[:, idx]
    weak_label_export[f"{label}__weight"] = diagnostic_weights[:, idx]
    weak_label_export[f"{label}__source"] = np.where(
        is_gold, "gold", report_statuses[f"{label}__status"]
    )
weak_label_export.to_csv(WORK_DIR / "weak_labels_v03.csv", index=False)

stats = []
for idx, label in enumerate(LABELS):
    statuses = report_statuses.loc[~is_gold, f"{label}__status"].value_counts()
    stats.append({
        "label": label,
        "gold_positive": int(train_df.loc[is_gold, label].sum()),
        "rule_positive": int(statuses.get("positive", 0)),
        "explicit_negative": int(statuses.get("explicit_negative", 0)),
        "unmentioned": int(statuses.get("unmentioned", 0)),
        "mean_soft_target": float(diagnostic_targets[~is_gold, idx].mean()),
        "mean_weak_weight": float(diagnostic_weights[~is_gold, idx].mean()),
    })

print(f"Gold studies: {is_gold.sum()} / {len(train_df)}")
print("Diagnostic calibration uses all gold rows for inspection only.")
display(pd.DataFrame(stats))
display(diagnostic_calibration)



## 3. Create study-level folds

Iterative multilabel stratification is used when it is available in the Kaggle environment.
Otherwise, the fallback stratifies by `is_gold`, distributing the 58 gold studies across
the five folds. Each study appears only once, preventing slice-level leakage between training
and validation.



In [ ]:
def assign_folds(y: np.ndarray, gold_mask: np.ndarray, n_splits: int, seed: int) -> np.ndarray:
    fold_ids = np.full(len(y), -1, dtype=np.int64)
    try:
        from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

        splitter = MultilabelStratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        strat_y = np.concatenate([y, gold_mask[:, None].astype(np.float32)], axis=1)
        splits = splitter.split(np.zeros(len(y)), strat_y)
        method = "MultilabelStratifiedKFold"
    except ImportError:
        splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        splits = splitter.split(np.zeros(len(y)), gold_mask.astype(np.int64))
        method = "StratifiedKFold(is_gold) fallback"

    for fold, (_, val_idx) in enumerate(splits):
        fold_ids[val_idx] = fold
    assert (fold_ids >= 0).all()
    print("Fold method:", method)
    return fold_ids


fold_ids = assign_folds(targets, is_gold, cfg.n_folds, cfg.seed)
train_df["fold"] = fold_ids
train_df["is_gold"] = is_gold

fold_summary = train_df.groupby("fold").agg(studies=(UID, "size"), gold=("is_gold", "sum"))
display(fold_summary)



## 4. Select three primary series for each study

Selection priority:

1. A series in the target plane with `Fluid_Sensitive=1` and `Fat_Suppression=1`.
2. Another fluid-sensitive or fat-suppressed series.
3. Any remaining series in the target plane.
4. Within the same priority, prefer a slice count close to the dataset median of 32 to avoid
   selecting unusually long 3D or high-resolution series.

All 4,407 training studies contain all three planes. A missing-plane mask is still implemented
so the pipeline remains robust to structural changes in the hidden test set. Building the
manifest requires one initial DICOM directory scan, and the result is cached in the working directory.



In [ ]:
def count_dicom_files(series_dir: Path) -> int:
    if not series_dir.exists():
        return 0
    return sum(1 for path in series_dir.iterdir() if path.is_file() and path.suffix.lower() == ".dcm")


def build_selected_series(
    studies: pd.DataFrame,
    series_frame: pd.DataFrame,
    image_root: Path,
    split: str,
) -> pd.DataFrame:
    cache_path = WORK_DIR / f"{split}_selected_series.csv"
    expected_ids = set(studies[UID].astype(str))
    if cache_path.exists():
        cached = pd.read_csv(cache_path)
        if set(cached[UID].astype(str)) == expected_ids:
            print("Using cached series selection:", cache_path)
            return cached

    grouped = {key: group for key, group in series_frame.groupby(UID, sort=False)}
    selected_rows = []

    for study_uid in tqdm(studies[UID].astype(str), desc=f"Select {split} series"):
        study_series = grouped.get(study_uid)
        record = {UID: study_uid}

        for plane in PLANES:
            candidates = [] if study_series is None else study_series[
                study_series["Anatomical_Plane"].eq(plane)
            ].to_dict("records")

            scored = []
            for candidate in candidates:
                series_uid = str(candidate["SeriesInstanceUID"])
                series_dir = image_root / study_uid / series_uid
                n_slices = count_dicom_files(series_dir)
                fluid = int(candidate.get("Fluid_Sensitive", 0))
                fat = int(candidate.get("Fat_Suppression", 0))
                both = int(fluid == 1 and fat == 1)
                score = (
                    int(n_slices > 0), both, fluid, fat,
                    -abs(n_slices - 32), n_slices, series_uid,
                )
                scored.append((score, candidate, n_slices))

            if scored:
                _, best, n_slices = max(scored, key=lambda item: item[0])
                record[f"{plane}_SeriesInstanceUID"] = str(best["SeriesInstanceUID"])
                record[f"{plane}_n_slices"] = int(n_slices)
                record[f"{plane}_Fluid_Sensitive"] = int(best.get("Fluid_Sensitive", 0))
                record[f"{plane}_Fat_Suppression"] = int(best.get("Fat_Suppression", 0))
            else:
                record[f"{plane}_SeriesInstanceUID"] = ""
                record[f"{plane}_n_slices"] = 0
                record[f"{plane}_Fluid_Sensitive"] = 0
                record[f"{plane}_Fat_Suppression"] = 0

        selected_rows.append(record)

    selected = pd.DataFrame(selected_rows)
    selected.to_csv(cache_path, index=False)
    return selected


if cfg.debug:
    # Select a deterministic subset while retaining as many gold-label studies as possible.
    gold_ids = train_df.loc[train_df["is_gold"], UID].tolist()
    remaining = train_df.loc[~train_df[UID].isin(gold_ids), UID].sample(
        n=max(0, cfg.debug_studies - len(gold_ids)), random_state=cfg.seed
    ).tolist()
    debug_ids = set(gold_ids + remaining)
    train_df = train_df[train_df[UID].isin(debug_ids)].reset_index(drop=True)
    debug_indices = weak_label_export[UID].isin(debug_ids).to_numpy()
    report_statuses = report_statuses.loc[debug_indices].reset_index(drop=True)
    reference_targets = reference_targets[debug_indices]
    targets = reference_targets.copy()
    is_gold = is_gold[debug_indices]
    print("DEBUG train studies:", len(train_df))

train_selected = build_selected_series(train_df, train_series_df, TRAIN_IMAGE_ROOT, "train_debug" if cfg.debug else "train")
test_selected = build_selected_series(test_df, test_series_df, TEST_IMAGE_ROOT, "test")

coverage = []
for plane in PLANES:
    coverage.append({
        "plane": plane,
        "train_available": int((train_selected[f"{plane}_n_slices"] > 0).sum()),
        "test_available": int((test_selected[f"{plane}_n_slices"] > 0).sum()),
        "train_preferred_FS": int(
            ((train_selected[f"{plane}_Fluid_Sensitive"] == 1) &
             (train_selected[f"{plane}_Fat_Suppression"] == 1)).sum()
        ),
    })
display(pd.DataFrame(coverage))
display(train_selected.head())



## 5. Build the slice order from DICOM spatial metadata

For each selected series:

- Prefer the projection of `ImagePositionPatient` onto the normal derived from
  `ImageOrientationPatient`.
- Fall back to `InstanceNumber` when spatial positions are incomplete.
- Use the filename only as the final deterministic fallback.

Approximately 400,000 headers from selected training slices are read once. The first run may
take several minutes. The ordered paths are cached as pickle files, so later epochs do not reread headers.



In [ ]:
ORDER_TAGS = ["ImageOrientationPatient", "ImagePositionPatient", "InstanceNumber"]


def safe_float_list(value: object, expected: int) -> Optional[np.ndarray]:
    try:
        array = np.asarray([float(v) for v in value], dtype=np.float64)
        return array if len(array) == expected else None
    except Exception:
        return None


def sort_dicom_files(series_dir: Path) -> Tuple[List[str], str, int]:
    files = sorted(
        path for path in series_dir.iterdir()
        if path.is_file() and path.suffix.lower() == ".dcm"
    ) if series_dir.exists() else []
    if not files:
        return [], "empty", 0

    records = []
    header_errors = 0
    for path in files:
        position_value = None
        instance_value = None
        try:
            ds = pydicom.dcmread(
                str(path), stop_before_pixels=True, force=True, specific_tags=ORDER_TAGS
            )
            orientation = safe_float_list(getattr(ds, "ImageOrientationPatient", None), 6)
            position = safe_float_list(getattr(ds, "ImagePositionPatient", None), 3)
            if orientation is not None and position is not None:
                normal = np.cross(orientation[:3], orientation[3:])
                norm = np.linalg.norm(normal)
                if norm > 0:
                    position_value = float(np.dot(position, normal / norm))
            if getattr(ds, "InstanceNumber", None) is not None:
                instance_value = float(ds.InstanceNumber)
        except Exception:
            header_errors += 1
        records.append((path, position_value, instance_value))

    positions = [record[1] for record in records]
    instances = [record[2] for record in records]
    if all(value is not None for value in positions) and len(set(positions)) == len(records):
        records.sort(key=lambda item: item[1])
        method = "ImagePositionPatient"
    elif all(value is not None for value in instances) and len(set(instances)) == len(records):
        records.sort(key=lambda item: item[2])
        method = "InstanceNumber"
    else:
        records.sort(key=lambda item: (
            item[2] is None,
            item[2] if item[2] is not None else math.inf,
            item[0].name,
        ))
        method = "mixed_fallback"

    return [str(record[0]) for record in records], method, header_errors


def build_ordered_path_cache(
    selected: pd.DataFrame,
    image_root: Path,
    split: str,
) -> Tuple[Dict[str, List[str]], pd.DataFrame]:
    cache_path = WORK_DIR / f"{split}_ordered_paths.pkl"
    audit_path = WORK_DIR / f"{split}_ordering_audit.csv"
    expected_series = {
        str(row[f"{plane}_SeriesInstanceUID"])
        for _, row in selected.iterrows()
        for plane in PLANES
        if pd.notna(row[f"{plane}_SeriesInstanceUID"]) and str(row[f"{plane}_SeriesInstanceUID"])
    }

    if cache_path.exists() and audit_path.exists():
        with open(cache_path, "rb") as file:
            cached = pickle.load(file)
        if expected_series.issubset(cached.keys()):
            print("Using cached DICOM ordering:", cache_path)
            return cached, pd.read_csv(audit_path)

    ordered_paths: Dict[str, List[str]] = {}
    audit_rows = []
    for _, row in tqdm(selected.iterrows(), total=len(selected), desc=f"Order {split} DICOM"):
        study_uid = str(row[UID])
        for plane in PLANES:
            value = row[f"{plane}_SeriesInstanceUID"]
            if pd.isna(value) or not str(value):
                continue
            series_uid = str(value)
            if series_uid in ordered_paths:
                continue
            paths, method, errors = sort_dicom_files(image_root / study_uid / series_uid)
            ordered_paths[series_uid] = paths
            audit_rows.append({
                UID: study_uid,
                "SeriesInstanceUID": series_uid,
                "plane": plane,
                "n_slices": len(paths),
                "sort_method": method,
                "header_errors": errors,
            })

    with open(cache_path, "wb") as file:
        pickle.dump(ordered_paths, file, protocol=pickle.HIGHEST_PROTOCOL)
    audit = pd.DataFrame(audit_rows)
    audit.to_csv(audit_path, index=False)
    return ordered_paths, audit


train_ordered_paths, train_order_audit = build_ordered_path_cache(
    train_selected, TRAIN_IMAGE_ROOT, "train_debug" if cfg.debug else "train"
)
test_ordered_paths, test_order_audit = build_ordered_path_cache(
    test_selected, TEST_IMAGE_ROOT, "test"
)

print("Train sorting methods:")
display(train_order_audit["sort_method"].value_counts(dropna=False).rename("series").to_frame())
print("Header read errors:", int(train_order_audit["header_errors"].sum()))



## 6. DICOM preprocessing and the 2.5D dataset

Six center positions are sampled uniformly from each plane. Each position uses
`[n-gap, n, n+gap]` as the three input channels. The three slices are jointly clipped at
the 1st and 99th percentiles and scaled to `[0, 1]`. The loader handles rescale slope/intercept,
`MONOCHROME1`, signed pixels, and a middle-frame fallback for multi-frame DICOM files.



In [ ]:
def read_dicom_pixels(path: str) -> Optional[np.ndarray]:
    try:
        ds = pydicom.dcmread(path, force=True)
        image = np.asarray(ds.pixel_array)
        image = np.squeeze(image)
        while image.ndim > 2:
            image = image[image.shape[0] // 2]
        if image.ndim != 2:
            return None
        image = image.astype(np.float32)
        slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
        intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
        image = image * slope + intercept
        if str(getattr(ds, "PhotometricInterpretation", "MONOCHROME2")) == "MONOCHROME1":
            image = image.max() + image.min() - image
        return image
    except Exception:
        return None


def sample_centers(n_slices: int, count: int) -> np.ndarray:
    if n_slices <= 1:
        return np.zeros(count, dtype=np.int64)
    # Avoid the outermost slices. Repeated centers are allowed for short series.
    return np.rint(np.linspace(0, n_slices - 1, count + 2)[1:-1]).astype(np.int64)


def normalize_triplet(images: Sequence[Optional[np.ndarray]], size: int) -> np.ndarray:
    valid = [image for image in images if image is not None and image.size > 0]
    if not valid:
        return np.zeros((size, size, 3), dtype=np.float32)

    values = np.concatenate([image.reshape(-1) for image in valid])
    low, high = np.percentile(values, [1.0, 99.0])
    if not np.isfinite(low) or not np.isfinite(high) or high <= low:
        low = float(np.min(values))
        high = float(np.max(values))
    if high <= low:
        high = low + 1.0

    fallback_shape = valid[0].shape
    channels = []
    for image in images:
        if image is None:
            image = np.zeros(fallback_shape, dtype=np.float32)
        image = np.clip(image, low, high)
        image = (image - low) / (high - low)
        image = cv2.resize(image, (size, size), interpolation=cv2.INTER_AREA)
        channels.append(image.astype(np.float32))
    return np.stack(channels, axis=-1)


def augment_triplet(image: np.ndarray) -> np.ndarray:
    if random.random() < 0.35:
        angle = random.uniform(-7.0, 7.0)
        scale = random.uniform(0.95, 1.05)
        center = (image.shape[1] / 2.0, image.shape[0] / 2.0)
        matrix = cv2.getRotationMatrix2D(center, angle, scale)
        image = cv2.warpAffine(
            image, matrix, (image.shape[1], image.shape[0]),
            flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101,
        )
        if image.ndim == 2:
            image = image[..., None]
    if random.random() < 0.50:
        gamma = random.uniform(0.85, 1.15)
        image = np.power(np.clip(image, 0, 1), gamma)
    if random.random() < 0.30:
        contrast = random.uniform(0.90, 1.10)
        brightness = random.uniform(-0.05, 0.05)
        image = np.clip(image * contrast + brightness, 0, 1)
    if random.random() < 0.20:
        noise = np.random.normal(0, random.uniform(0.005, 0.02), image.shape)
        image = np.clip(image + noise, 0, 1)
    return image.astype(np.float32)


IMAGENET_MEAN = np.asarray([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.asarray([0.229, 0.224, 0.225], dtype=np.float32)


class KneeStudyDataset(Dataset):
    def __init__(
        self,
        frame: pd.DataFrame,
        selected: pd.DataFrame,
        ordered_paths: Dict[str, List[str]],
        targets_array: Optional[np.ndarray] = None,
        weights_array: Optional[np.ndarray] = None,
        training: bool = False,
    ) -> None:
        self.frame = frame.reset_index(drop=True).copy()
        self.selected = selected.set_index(UID)
        self.ordered_paths = ordered_paths
        self.targets = targets_array
        self.weights = weights_array
        self.training = training
        if targets_array is not None:
            assert len(targets_array) == len(self.frame)

    def __len__(self) -> int:
        return len(self.frame)

    def _load_plane(self, paths: List[str]) -> torch.Tensor:
        output = torch.zeros(
            cfg.samples_per_plane, 3, cfg.image_size, cfg.image_size,
            dtype=torch.float32,
        )
        if not paths:
            return output

        centers = sample_centers(len(paths), cfg.samples_per_plane)
        pixel_cache: Dict[str, Optional[np.ndarray]] = {}
        for sample_idx, center in enumerate(centers):
            indices = np.clip(
                [center - cfg.triplet_gap, center, center + cfg.triplet_gap],
                0, len(paths) - 1,
            )
            images = []
            for index in indices:
                path = paths[int(index)]
                if path not in pixel_cache:
                    pixel_cache[path] = read_dicom_pixels(path)
                images.append(pixel_cache[path])

            triplet = normalize_triplet(images, cfg.image_size)
            if self.training:
                triplet = augment_triplet(triplet)
            triplet = (triplet - IMAGENET_MEAN) / IMAGENET_STD
            output[sample_idx] = torch.from_numpy(triplet.transpose(2, 0, 1).copy())
        return output

    def __getitem__(self, index: int) -> Dict[str, object]:
        row = self.frame.iloc[index]
        study_uid = str(row[UID])
        selected_row = self.selected.loc[study_uid]

        plane_images = []
        plane_mask = []
        sequence_meta = []
        for plane in PLANES:
            value = selected_row[f"{plane}_SeriesInstanceUID"]
            series_uid = "" if pd.isna(value) else str(value)
            paths = self.ordered_paths.get(series_uid, []) if series_uid else []
            valid = float(len(paths) > 0)
            plane_images.append(self._load_plane(paths))
            plane_mask.append(valid)
            sequence_meta.append([
                float(selected_row[f"{plane}_Fluid_Sensitive"]) if valid else 0.0,
                float(selected_row[f"{plane}_Fat_Suppression"]) if valid else 0.0,
            ])

        item: Dict[str, object] = {
            UID: study_uid,
            "images": torch.stack(plane_images),       # [P, K, 3, H, W]
            "plane_mask": torch.tensor(plane_mask, dtype=torch.float32),
            "sequence_meta": torch.tensor(sequence_meta, dtype=torch.float32),
            "is_gold": torch.tensor(bool(row.get("is_gold", False))),
        }
        if self.targets is not None:
            item["targets"] = torch.from_numpy(self.targets[index].astype(np.float32))
            item["target_weights"] = torch.from_numpy(self.weights[index].astype(np.float32))
        return item


def preflight_dicom_check(path_cache: Dict[str, List[str]], n: int = 24) -> None:
    candidates = [paths[len(paths) // 2] for paths in path_cache.values() if paths]
    rng = random.Random(cfg.seed)
    rng.shuffle(candidates)
    candidates = candidates[:min(n, len(candidates))]
    failures = sum(read_dicom_pixels(path) is None for path in tqdm(candidates, desc="DICOM preflight"))
    print(f"DICOM preflight failures: {failures}/{len(candidates)}")
    if candidates and failures / len(candidates) > 0.10:
        raise RuntimeError(
            "More than 10% of sampled DICOM files could not be decoded. "
            "Check the pydicom decoding plugins and the competition data."
        )


preflight_dicom_check(train_ordered_paths)



## 7. Shared EfficientNet model for three planes

All 2.5D triplets pass through the same backbone. Features from the six positions in each plane
are mean-pooled, then the three plane features, plane masks, and sequence metadata are concatenated
before producing 12 logits.



In [ ]:
class MultiPlaneEfficientNet(nn.Module):
    def __init__(self, pretrained: bool = True) -> None:
        super().__init__()
        self.feature_dim = 1280
        self.backbone_source = ""

        try:
            import timm

            use_pretrained = pretrained and cfg.local_backbone_weights is None
            try:
                self.backbone = timm.create_model(
                    cfg.backbone, pretrained=use_pretrained, num_classes=0, global_pool="avg"
                )
            except Exception as error:
                print(f"timm pretrained weights unavailable ({error}); using random initialization.")
                self.backbone = timm.create_model(
                    cfg.backbone, pretrained=False, num_classes=0, global_pool="avg"
                )
            self.feature_dim = int(self.backbone.num_features)
            self.backbone_source = "timm"
        except ImportError:
            from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0

            weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
            try:
                model = efficientnet_b0(weights=weights)
            except Exception as error:
                print(f"torchvision pretrained weights unavailable ({error}); using random initialization.")
                model = efficientnet_b0(weights=None)
            self.feature_dim = int(model.classifier[1].in_features)
            model.classifier = nn.Identity()
            self.backbone = model
            self.backbone_source = "torchvision"

        if cfg.local_backbone_weights:
            state = torch.load(cfg.local_backbone_weights, map_location="cpu")
            state = state.get("state_dict", state)
            state = {key.removeprefix("module."): value for key, value in state.items()}
            missing, unexpected = self.backbone.load_state_dict(state, strict=False)
            print("Loaded local backbone weights; missing/unexpected:", len(missing), len(unexpected))

        metadata_dim = len(PLANES) * 3  # valid mask + fluid sensitive + fat suppression
        fusion_dim = self.feature_dim * len(PLANES) + metadata_dim
        hidden_dim = 512
        self.head = nn.Sequential(
            nn.LayerNorm(fusion_dim),
            nn.Linear(fusion_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(hidden_dim, len(LABELS)),
        )

    def forward(
        self,
        images: torch.Tensor,
        plane_mask: torch.Tensor,
        sequence_meta: torch.Tensor,
    ) -> torch.Tensor:
        batch_size, n_planes, n_samples, channels, height, width = images.shape
        images = images.reshape(batch_size * n_planes * n_samples, channels, height, width)
        features = self.backbone(images)
        if features.ndim > 2:
            features = F.adaptive_avg_pool2d(features, 1).flatten(1)
        features = features.reshape(batch_size, n_planes, n_samples, self.feature_dim)
        plane_features = features.mean(dim=2) * plane_mask.unsqueeze(-1)

        metadata = torch.cat([plane_mask.unsqueeze(-1), sequence_meta], dim=-1).flatten(1)
        fused = torch.cat([plane_features.flatten(1), metadata], dim=1)
        return self.head(fused)


model_smoke = MultiPlaneEfficientNet(pretrained=False).to(DEVICE)
with torch.no_grad():
    smoke_logits = model_smoke(
        torch.zeros(1, len(PLANES), cfg.samples_per_plane, 3, cfg.image_size, cfg.image_size, device=DEVICE),
        torch.ones(1, len(PLANES), device=DEVICE),
        torch.ones(1, len(PLANES), 2, device=DEVICE),
    )
assert smoke_logits.shape == (1, len(LABELS))
print("Model smoke test:", smoke_logits.shape, model_smoke.backbone_source)
del model_smoke, smoke_logits
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()



## 8. Training, validation, runtime protection, and metrics

V03 uses confidence-weighted BCE without class `pos_weight`. Weak targets are calibrated
separately inside each fold, using only that fold's training gold rows.

Kaggle limits a Notebook run to nine hours. The runtime guard checks the budget at optimizer
boundaries and reserves 35 minutes for checkpoint serialization and Notebook shutdown. If the
guard triggers, the current model, optimizer, scheduler, AMP scaler, RNG states, fold, and epoch
are written to `v03_fold_<fold>_last.pt`. Remaining validation and inference are skipped so the
Notebook can finish normally and preserve its output. Attach that output to the next run; V03
finds the last checkpoint automatically, or set `cfg.resume_checkpoint` explicitly.



In [ ]:
def make_loader(dataset: Dataset, shuffle: bool) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=cfg.batch_size,
        shuffle=shuffle,
        num_workers=cfg.num_workers,
        pin_memory=AMP_ENABLED,
        persistent_workers=cfg.num_workers > 0,
        drop_last=False,
    )


def runtime_seconds_remaining() -> float:
    usable_seconds = (
        cfg.runtime_limit_hours * 3600.0 - cfg.runtime_safety_margin_minutes * 60.0
    )
    return usable_seconds - (time.monotonic() - NOTEBOOK_STARTED_AT)


def runtime_stop_due() -> bool:
    return runtime_seconds_remaining() <= 0.0


def format_runtime(seconds: float) -> str:
    seconds = max(0.0, seconds)
    return f"{int(seconds // 3600):02d}:{int((seconds % 3600) // 60):02d}:{int(seconds % 60):02d}"


def weighted_bce_loss(
    logits: torch.Tensor,
    targets_tensor: torch.Tensor,
    weights_tensor: torch.Tensor,
) -> torch.Tensor:
    """Confidence-weighted BCE without the V01 class pos_weight."""
    raw = F.binary_cross_entropy_with_logits(logits, targets_tensor, reduction="none")
    return (raw * weights_tensor).sum() / weights_tensor.sum().clamp_min(1.0)


def move_batch(batch: Dict[str, object]) -> Tuple[torch.Tensor, ...]:
    return (
        batch["images"].to(DEVICE, non_blocking=True),
        batch["plane_mask"].to(DEVICE, non_blocking=True),
        batch["sequence_meta"].to(DEVICE, non_blocking=True),
        batch["targets"].to(DEVICE, non_blocking=True),
        batch["target_weights"].to(DEVICE, non_blocking=True),
    )


def capture_rng_state() -> Dict[str, object]:
    state: Dict[str, object] = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state["cuda"] = torch.cuda.get_rng_state_all()
    return state


def restore_rng_state(state: Optional[Dict[str, object]]) -> None:
    if not state:
        return
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"].cpu())
    if torch.cuda.is_available() and "cuda" in state:
        torch.cuda.set_rng_state_all([s.cpu() for s in state["cuda"]])


def save_training_state(
    path: Path,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler.LRScheduler,
    scaler: torch.cuda.amp.GradScaler,
    fold: int,
    resume_epoch: int,
    best_val_loss: float,
    epochs_without_improvement: int,
    reason: str,
) -> None:
    torch.save({
        "checkpoint_type": "training_state",
        "version": cfg.version,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "rng_state": capture_rng_state(),
        "fold": fold,
        "resume_epoch": resume_epoch,
        "best_val_loss": best_val_loss,
        "epochs_without_improvement": epochs_without_improvement,
        "reason": reason,
        "labels": LABELS,
        "config": asdict(cfg),
    }, path)
    print(f"Saved resumable checkpoint ({reason}): {path}")


def checkpoint_candidates(fold: int, kind: str) -> List[Path]:
    filename = f"{cfg.version}_fold_{fold}_{kind}.pt"
    candidates = []
    local = WORK_DIR / filename
    if local.exists():
        candidates.append(local)
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        candidates.extend(kaggle_input.rglob(filename))
    return sorted(set(candidates), key=lambda path: path.stat().st_mtime, reverse=True)


def find_resume_checkpoint(fold: int) -> Optional[Path]:
    if cfg.resume_checkpoint:
        explicit = Path(cfg.resume_checkpoint)
        if not explicit.exists():
            raise FileNotFoundError(f"Configured resume checkpoint does not exist: {explicit}")
        return explicit
    if not cfg.auto_resume:
        return None
    candidates = checkpoint_candidates(fold, "last")
    return candidates[0] if candidates else None


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scaler: torch.cuda.amp.GradScaler,
) -> Tuple[float, bool]:
    model.train()
    optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0
    seen = 0

    progress = tqdm(loader, desc="train", leave=False)
    for step, batch in enumerate(progress):
        images, plane_mask, sequence_meta, y, weights = move_batch(batch)
        with torch.cuda.amp.autocast(enabled=AMP_ENABLED):
            logits = model(images, plane_mask, sequence_meta)
            loss = weighted_bce_loss(logits, y, weights)
            scaled_loss = loss / cfg.accumulation_steps
        scaler.scale(scaled_loss).backward()

        should_step = (step + 1) % cfg.accumulation_steps == 0 or (step + 1) == len(loader)
        if should_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        batch_size = images.size(0)
        running_loss += float(loss.detach()) * batch_size
        seen += batch_size
        progress.set_postfix(
            loss=f"{running_loss / max(seen, 1):.4f}",
            remaining=format_runtime(runtime_seconds_remaining()),
        )
        if should_step and runtime_stop_due():
            return running_loss / max(seen, 1), True

    return running_loss / max(seen, 1), False


@torch.no_grad()
def validate(
    model: nn.Module,
    loader: DataLoader,
) -> Tuple[float, np.ndarray, np.ndarray, np.ndarray, List[str]]:
    model.eval()
    losses, probabilities, truths, gold_flags, study_ids = [], [], [], [], []
    for batch in tqdm(loader, desc="valid", leave=False):
        images, plane_mask, sequence_meta, y, weights = move_batch(batch)
        with torch.cuda.amp.autocast(enabled=AMP_ENABLED):
            logits = model(images, plane_mask, sequence_meta)
            loss = weighted_bce_loss(logits, y, weights)
        losses.append(float(loss) * images.size(0))
        probabilities.append(torch.sigmoid(logits).cpu().numpy())
        truths.append(y.cpu().numpy())
        gold_flags.append(batch["is_gold"].numpy().astype(bool))
        study_ids.extend(list(batch[UID]))

    return (
        sum(losses) / len(loader.dataset),
        np.concatenate(probabilities),
        np.concatenate(truths),
        np.concatenate(gold_flags),
        study_ids,
    )


def metric_table(y_true: np.ndarray, probability: np.ndarray) -> pd.DataFrame:
    rows = []
    probability = np.clip(probability, 1e-6, 1 - 1e-6)
    for idx, label in enumerate(LABELS):
        target = y_true[:, idx]
        auc = np.nan
        if len(np.unique(target)) == 2:
            auc = roc_auc_score(target, probability[:, idx])
        rows.append({
            "label": label,
            "n": len(target),
            "positive": int(target.sum()),
            "auc": auc,
            "log_loss": log_loss(target, probability[:, idx], labels=[0, 1]),
        })
    return pd.DataFrame(rows)


checkpoint_paths = []
oof_rows = []
RUNTIME_STOP_REQUESTED = False

for fold in cfg.folds_to_train:
    print(f"\n{'=' * 20} FOLD {fold} {'=' * 20}")
    train_indices = np.flatnonzero(train_df["fold"].to_numpy() != fold)
    val_indices = np.flatnonzero(train_df["fold"].to_numpy() == fold)
    fold_train = train_df.iloc[train_indices].reset_index(drop=True)
    fold_val = train_df.iloc[val_indices].reset_index(drop=True)

    fold_targets, fold_weights, fold_calibration = build_fold_targets(
        train_df, report_statuses, train_indices
    )
    fold_calibration.insert(0, "fold", fold)
    calibration_path = WORK_DIR / f"{cfg.version}_fold_{fold}_rule_calibration.csv"
    fold_calibration.to_csv(calibration_path, index=False)
    assert_calibration_valid(fold_calibration)
    y_train, w_train = fold_targets[train_indices], fold_weights[train_indices]
    y_val, w_val = fold_targets[val_indices], fold_weights[val_indices]

    print(f"Fold calibration saved: {calibration_path}")
    print(
        "Runtime available before training:",
        format_runtime(runtime_seconds_remaining()),
        f"(safety margin={cfg.runtime_safety_margin_minutes:.0f} minutes)",
    )

    train_dataset = KneeStudyDataset(
        fold_train, train_selected, train_ordered_paths,
        y_train, w_train, training=True,
    )
    val_dataset = KneeStudyDataset(
        fold_val, train_selected, train_ordered_paths,
        y_val, w_val, training=False,
    )
    train_loader = make_loader(train_dataset, shuffle=True)
    val_loader = make_loader(val_dataset, shuffle=False)

    model = MultiPlaneEfficientNet(pretrained=cfg.pretrained).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(1, cfg.epochs), eta_min=cfg.learning_rate * 0.05
    )
    scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)

    best_checkpoint_path = WORK_DIR / f"{cfg.version}_fold_{fold}_best.pt"
    last_checkpoint_path = WORK_DIR / f"{cfg.version}_fold_{fold}_last.pt"
    best_loss = math.inf
    best_reference_auc = -math.inf
    best_auc_checkpoint_path = WORK_DIR / f"{cfg.version}_fold_{fold}_best_reference_auc.pt"
    epochs_without_improvement = 0
    start_epoch = 0

    resume_path = find_resume_checkpoint(fold)
    if resume_path is not None:
        # Training-state checkpoints are trusted artifacts created by this Notebook.
        # weights_only=False is required for the serialized NumPy RNG state on PyTorch 2.6+.
        resume_state = torch.load(
            resume_path, map_location=DEVICE, weights_only=False
        )
        if resume_state.get("version") != cfg.version or int(resume_state.get("fold", -1)) != fold:
            raise ValueError(f"Resume checkpoint does not match {cfg.version} fold {fold}: {resume_path}")
        model.load_state_dict(resume_state["model"])
        optimizer.load_state_dict(resume_state["optimizer"])
        scheduler.load_state_dict(resume_state["scheduler"])
        saved_scaler_state = resume_state.get("scaler")
        if saved_scaler_state:
            scaler.load_state_dict(saved_scaler_state)
        start_epoch = int(resume_state.get("resume_epoch", 0))
        best_loss = float(resume_state.get("best_val_loss", math.inf))
        epochs_without_improvement = int(resume_state.get("epochs_without_improvement", 0))
        restore_rng_state(resume_state.get("rng_state"))
        if best_auc_checkpoint_path.exists():
            try:
                best_reference_auc = float(torch.load(
                    best_auc_checkpoint_path, map_location="cpu"
                ).get("best_reference_auc", -math.inf))
            except Exception:
                best_reference_auc = -math.inf
        print(
            f"Resumed {cfg.version} fold {fold} from {resume_path}; "
            f"next epoch index={start_epoch}, best_val_loss={best_loss:.6f}"
        )

    for epoch in range(start_epoch, cfg.epochs):
        if runtime_stop_due():
            save_training_state(
                last_checkpoint_path, model, optimizer, scheduler, scaler,
                fold, epoch, best_loss, epochs_without_improvement, "time_limit_before_epoch",
            )
            RUNTIME_STOP_REQUESTED = True
            break

        started = time.time()
        train_loss, stopped_during_epoch = train_one_epoch(
            model, train_loader, optimizer, scaler
        )
        if stopped_during_epoch:
            # The current epoch is intentionally restarted on resume because the shuffled
            # DataLoader sampler position is not serialized.
            save_training_state(
                last_checkpoint_path, model, optimizer, scheduler, scaler,
                fold, epoch, best_loss, epochs_without_improvement, "time_limit_during_epoch",
            )
            RUNTIME_STOP_REQUESTED = True
            break

        val_loss, val_probability, val_truth, val_gold, val_ids = validate(model, val_loader)
        scheduler.step()
        minutes = (time.time() - started) / 60
        all_metrics = metric_table(reference_targets[val_indices], val_probability)
        macro_auc = all_metrics["auc"].mean()
        print(
            f"epoch={epoch + 1}/{cfg.epochs} train_loss={train_loss:.5f} "
            f"val_loss={val_loss:.5f} reference_macro_auc={macro_auc:.5f} "
            f"lr={optimizer.param_groups[0]['lr']:.2e} time={minutes:.1f}m "
            f"remaining={format_runtime(runtime_seconds_remaining())}"
        )

        if val_loss < best_loss:
            best_loss = val_loss
            epochs_without_improvement = 0
            torch.save({
                "checkpoint_type": "best_model",
                "version": cfg.version,
                "model": model.state_dict(),
                "fold": fold,
                "epoch": epoch,
                "best_val_loss": best_loss,
                "labels": LABELS,
                "config": asdict(cfg),
            }, best_checkpoint_path)
            print("Saved best checkpoint:", best_checkpoint_path)
        else:
            epochs_without_improvement += 1

        if macro_auc > best_reference_auc:
            best_reference_auc = float(macro_auc)
            torch.save({
                "checkpoint_type": "best_reference_auc",
                "version": cfg.version,
                "model": model.state_dict(),
                "fold": fold,
                "epoch": epoch,
                "best_val_loss": float(val_loss),
                "best_reference_auc": best_reference_auc,
                "labels": LABELS,
                "config": asdict(cfg),
            }, best_auc_checkpoint_path)
            print("Saved best reference-AUC checkpoint:", best_auc_checkpoint_path)

        save_training_state(
            last_checkpoint_path, model, optimizer, scheduler, scaler,
            fold, epoch + 1, best_loss, epochs_without_improvement, "epoch_complete",
        )

        if runtime_stop_due():
            RUNTIME_STOP_REQUESTED = True
            print("Runtime safety boundary reached after the completed epoch.")
            break
        if epochs_without_improvement >= cfg.early_stopping_patience:
            print("Metric early stopping")
            break

    if RUNTIME_STOP_REQUESTED:
        print(
            "Training stopped safely near the Kaggle runtime limit. "
            "Attach this Notebook output to the next run to auto-resume."
        )
        del model, optimizer, scheduler, scaler, train_loader, val_loader
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        break

    best_candidates = checkpoint_candidates(fold, "best")
    if not best_candidates:
        raise FileNotFoundError(
            f"No best checkpoint exists for completed {cfg.version} fold {fold}."
        )
    selected_best_path = best_candidates[0]
    checkpoint_paths.append(selected_best_path)
    checkpoint = torch.load(selected_best_path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model"])
    val_loss, val_probability, val_truth, val_gold, val_ids = validate(model, val_loader)

    fold_metrics = metric_table(reference_targets[val_indices], val_probability)
    print("Binary rule-reference validation labels:")
    display(fold_metrics)
    if val_gold.any():
        print(f"Gold validation subset: {val_gold.sum()} studies")
        display(metric_table(val_truth[val_gold], val_probability[val_gold]))

    fold_oof = pd.DataFrame({UID: val_ids, "fold": fold, "is_gold": val_gold})
    for idx, label in enumerate(LABELS):
        fold_oof[f"{label}__soft_target"] = val_truth[:, idx]
        fold_oof[f"{label}__reference_target"] = reference_targets[val_indices, idx]
        fold_oof[f"{label}__pred"] = val_probability[:, idx]
    oof_rows.append(fold_oof)

    # Prediction-spread diagnostic. Near-zero std means the model collapsed to a
    # per-label constant (the V02 failure). This is a collapse alarm only, never an
    # optimization target: a random model can also have a large spread.
    pred_std = val_probability.std(axis=0)
    print("Prediction-spread diagnostic (per-label std of best-soft-loss predictions):")
    display(pd.DataFrame({
        "label": LABELS,
        "pred_mean": val_probability.mean(axis=0),
        "pred_std": pred_std,
    }))
    if float(pred_std.mean()) < 0.03:
        print(
            "WARNING: mean prediction std < 0.03 -- predictions are near-constant. "
            "The supervision signal likely collapsed to the base rate (see V02)."
        )

    # Report the best reference-AUC checkpoint alongside the primary best-soft-loss
    # model. The reference AUC is scored against binarized rule labels, so it favours
    # discrimination but is biased toward agreement with the noisy weak labels; treat
    # it as a diagnostic, not the final selection metric.
    if best_auc_checkpoint_path.exists():
        auc_checkpoint = torch.load(best_auc_checkpoint_path, map_location=DEVICE)
        model.load_state_dict(auc_checkpoint["model"])
        _, auc_probability, auc_truth, auc_gold, _ = validate(model, val_loader)
        print("Best reference-AUC checkpoint -- rule-reference validation labels:")
        display(metric_table(reference_targets[val_indices], auc_probability))
        if auc_gold.any():
            print("Best reference-AUC checkpoint -- gold validation subset:")
            display(metric_table(auc_truth[auc_gold], auc_probability[auc_gold]))
        print(
            "Best reference-AUC checkpoint mean prediction std:",
            f"{float(auc_probability.std(axis=0).mean()):.4f}",
        )

    del model, optimizer, scheduler, scaler, train_loader, val_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if oof_rows:
    oof = pd.concat(oof_rows, ignore_index=True)
    oof_path = WORK_DIR / f"oof_predictions_{cfg.version}.csv"
    oof.to_csv(oof_path, index=False)
    print("OOF:", oof_path)
else:
    print("OOF generation skipped because training stopped at the runtime boundary.")
print("Saved best checkpoints:", [str(path) for path in checkpoint_paths])



## 9. Test inference and submission.csv

Predictions are averaged across all checkpoints trained in the current run.
The submission retains continuous probabilities and is not thresholded to binary values.



In [ ]:
@torch.no_grad()
def predict(model: nn.Module, loader: DataLoader) -> Tuple[np.ndarray, List[str]]:
    model.eval()
    predictions, study_ids = [], []
    for batch in tqdm(loader, desc="test inference"):
        images = batch["images"].to(DEVICE, non_blocking=True)
        plane_mask = batch["plane_mask"].to(DEVICE, non_blocking=True)
        sequence_meta = batch["sequence_meta"].to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=AMP_ENABLED):
            logits = model(images, plane_mask, sequence_meta)
        predictions.append(torch.sigmoid(logits).cpu().numpy())
        study_ids.extend(list(batch[UID]))
    return np.concatenate(predictions), study_ids


def run_test_inference() -> None:
    test_dataset = KneeStudyDataset(
        test_df.assign(is_gold=False), test_selected, test_ordered_paths, training=False
    )
    test_loader = make_loader(test_dataset, shuffle=False)

    inference_checkpoints = list(checkpoint_paths)
    if not inference_checkpoints:
        for fold in cfg.folds_to_train:
            candidates = checkpoint_candidates(fold, "best")
            if candidates:
                inference_checkpoints.append(candidates[0])
    if not inference_checkpoints:
        raise FileNotFoundError(f"No {cfg.version} best checkpoint was found; test inference cannot run.")

    ensemble = []
    inference_ids = None
    for checkpoint_path in inference_checkpoints:
        checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
        model = MultiPlaneEfficientNet(pretrained=False).to(DEVICE)
        model.load_state_dict(checkpoint["model"])
        probability, current_ids = predict(model, test_loader)
        if inference_ids is None:
            inference_ids = current_ids
        else:
            assert inference_ids == current_ids
        ensemble.append(probability)
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    test_probability = np.mean(ensemble, axis=0)
    prediction_frame = pd.DataFrame(test_probability, columns=LABELS)
    prediction_frame.insert(0, UID, inference_ids)

    submission = sample_submission[[UID]].merge(
        prediction_frame, on=UID, how="left", validate="one_to_one"
    )
    assert submission[LABELS].notna().all().all(), "At least one test study has no prediction"
    submission[LABELS] = submission[LABELS].clip(1e-6, 1 - 1e-6)

    submission_path = Path("/kaggle/working/submission.csv")
    if not submission_path.parent.exists():
        submission_path = WORK_DIR / "submission.csv"
    submission.to_csv(submission_path, index=False)
    submission.to_csv(WORK_DIR / f"submission_{cfg.version}.csv", index=False)

    print("Submission:", submission_path)
    print("Shape:", submission.shape)
    display(submission.head())


if RUNTIME_STOP_REQUESTED:
    print(
        "Test inference skipped to preserve the resumable checkpoint before Kaggle's runtime limit."
    )
else:
    run_test_inference()



## 10. Output artifacts

The following files are generated in `WORK_DIR`:

```text
weak_labels_v03.csv
v03_rule_calibration_diagnostic.csv
v03_fold_{0..4}_rule_calibration.csv
train_selected_series.csv
test_selected_series.csv
train_ordered_paths.pkl
test_ordered_paths.pkl
train_ordering_audit.csv
test_ordering_audit.csv
v03_fold_{0..4}_best.pt              # per-fold best soft-loss weights (ensembled for submission)
v03_fold_{0..4}_best_reference_auc.pt # per-fold best rule-reference-AUC model (diagnostic)
v03_fold_{0..4}_last.pt              # per-fold resumable training state
oof_predictions_v03.csv
submission_v03.csv
```

Recommended run sequence:

1. Keep `debug=False` and `folds_to_train=(0, 1, 2, 3, 4)` for the full five-fold training and inference run.
2. Set `batch_size` to 1 or 2 based on GPU memory. Reduce `image_size` from 320 to 288 if needed.
3. Five folds will not finish in one Kaggle session: when the runtime limit is hit the notebook saves a resumable `*_last.pt` and stops before inference. Attach the output as input and re-run to auto-resume; submission and OOF are written only after all five folds complete.
4. Download the checkpoints, OOF predictions, weak labels, and submission file.
5. Improve report weak labels first, then test attention pooling, a larger backbone, or multi-series input.
